In [2]:
import argparse
import re
import pandas as pd

MONTH_ID = {
    "januari": 1, "februari": 2, "maret": 3, "april": 4, "mei": 5, "juni": 6,
    "juli": 7, "agustus": 8, "september": 9, "oktober": 10, "november": 11, "desember": 12,
}

def _strip_bom_and_spaces(cols):
    return (
        pd.Index(cols).astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

def _find_column_case_insensitive(df, target_lower: str):
    for c in df.columns:
        if str(c).strip().lower() == target_lower:
            return c
    return None

def _parse_month_to_int(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    m1 = pd.to_datetime(s, format="%B", errors="coerce").dt.month  # English month name
    s_lower = s.str.lower()
    m2 = s_lower.map(MONTH_ID).astype("Float64")  # Indonesian month name
    out = m1.astype("Float64").fillna(m2)
    return out.astype("Int64")

def _parse_decimal_comma_number(series: pd.Series) -> pd.Series:
    # "108,54" -> 108.54 ; robust kalau ada pemisah ribuan
    s = series.astype(str).str.strip()
    s = s.str.replace(r"[\u00A0\s]", "", regex=True)  # remove spaces
    s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def preprocess_ihk(ihk_path: str, region: str = "DKI", item: str = "makanan") -> pd.DataFrame:
    ihk = pd.read_csv(ihk_path, sep=";")
    ihk.columns = _strip_bom_and_spaces(ihk.columns)

    bulan_col = _find_column_case_insensitive(ihk, "bulan")
    if not bulan_col:
        raise ValueError(f"Kolom 'Bulan' tidak ketemu. Kolom yang ada: {list(ihk.columns)}")

    year_cols = [c for c in ihk.columns if re.fullmatch(r"\d{4}", str(c).strip())]
    if not year_cols:
        raise ValueError("Tidak menemukan kolom tahun (mis. 2024, 2025). "
                         f"Kolom yang ada: {list(ihk.columns)}")

    ihk = ihk[[bulan_col] + year_cols].copy()

    # wide -> long
    ihk_long = ihk.melt(id_vars=[bulan_col], var_name="year", value_name="ihk_raw")
    ihk_long.rename(columns={bulan_col: "Bulan"}, inplace=True)

    ihk_long["year"] = pd.to_numeric(ihk_long["year"], errors="coerce").astype("Int64")
    ihk_long["month"] = _parse_month_to_int(ihk_long["Bulan"])
    ihk_long["ihk"] = _parse_decimal_comma_number(ihk_long["ihk_raw"])

    ihk_long["date_month"] = pd.to_datetime(
        dict(year=ihk_long["year"], month=ihk_long["month"], day=1),
        errors="coerce"
    )

    ihk_long["region"] = region
    ihk_long["ihk_item"] = item
    ihk_long = ihk_long.drop(columns=["ihk_raw"]).sort_values(["region","ihk_item","date_month"])

    # kalender bulanan lengkap (tanpa groupby.apply)
    min_dt = ihk_long["date_month"].min()
    max_dt = ihk_long["date_month"].max()
    if pd.isna(min_dt) or pd.isna(max_dt):
        raise ValueError("date_month kosong/NaT semua. Cek parsing 'Bulan' dan kolom tahun.")

    full_months = pd.date_range(min_dt, max_dt, freq="MS")
    base = pd.DataFrame({
        "region": [region] * len(full_months),
        "ihk_item": [item] * len(full_months),
        "date_month": full_months
    })

    ihk_long = base.merge(ihk_long, on=["region","ihk_item","date_month"], how="left")

    # restore year/month/Bulan kalau ada missing
    ihk_long["year"]  = ihk_long["year"].fillna(ihk_long["date_month"].dt.year).astype("Int64")
    ihk_long["month"] = ihk_long["month"].fillna(ihk_long["date_month"].dt.month).astype("Int64")
    ihk_long["Bulan"] = ihk_long["Bulan"].fillna(ihk_long["date_month"].dt.strftime("%B"))

    # fitur inflasi
    ihk_long = ihk_long.sort_values(["region","ihk_item","date_month"])
    ihk_long["ihk_mom_pct"] = ihk_long.groupby(["region","ihk_item"])["ihk"].pct_change() * 100
    ihk_long["ihk_yoy_pct"] = ihk_long.groupby(["region","ihk_item"])["ihk"].pct_change(12) * 100

    cols = ["region","ihk_item","date_month","year","month","Bulan","ihk","ihk_mom_pct","ihk_yoy_pct"]
    return ihk_long[cols].reset_index(drop=True)

# ==== RUN (langsung) ====
INFILE  = "IHK.2024-2025.Makanan.csv"
OUTFILE = "ihk_makanan_clean_long.csv"

out = preprocess_ihk(INFILE, region="DKI", item="makanan")
out.to_csv(OUTFILE, index=False)

print("✅ Saved:", OUTFILE)
print("Rows:", len(out), "| Range:", out["date_month"].min().date(), "->", out["date_month"].max().date())
out.head()

✅ Saved: ihk_makanan_clean_long.csv
Rows: 24 | Range: 2024-01-01 -> 2025-12-01


,region,ihk_item,date_month,year,month,Bulan,ihk,ihk_mom_pct,ihk_yoy_pct
0,DKI,makanan,2024-01-01,2024,1,January,108.54,NaN,NaN
1,DKI,makanan,2024-02-01,2024,2,February,109.63,1.004238,NaN
2,DKI,makanan,2024-03-01,2024,3,March,111.19,1.422968,NaN
3,DKI,makanan,2024-04-01,2024,4,April,111.16,-0.026981,NaN
4,DKI,makanan,2024-05-01,2024,5,May,110.84,-0.287873,NaN
